## **EPSCs AND IPSCs ANALYSIS**

# IMPORTING DATA
This section install the library to read .abf files

Follow this directions
*   Upload the abf file in the folder menu on the left
*   Right-click on the file name and copy the path
*   Paste the path on the third line "xxx.abf"
*   Run the section







In [ ]:
!pip install pyabf
!pip install matplotlib

import pyabf  #to be able to import abf files
import matplotlib.pyplot as plt

registro = pyabf.ABF("/content/record contin.abf")
#print(registro)
FreqMuestreo = registro.dataRate # This is the sampling frequency in Hz

registro.setSweep(0) # Sweep 0 is the signal
plt.figure
plt.figure(figsize=(12, 8))
plt.plot(registro.sweepX, registro.sweepY) # Corrected to sweepX and sweepY
plt.title("Continuos Recording - Em=-40")
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.show()

# Filtering the signal
Filtering the signal with a 4th-order Butterworth low-pass fillter with zero-phase filtering

In the slide bar select the Loss-Pass filter Cutoff frequency, the standard is 500Hz (could go to 1KHz)

Run the section to display the result.



In [ ]:
#import numpy as np ##Importing the library numpy
from scipy import signal ##SciPy is the library python library for signals
import matplotlib.pyplot as plt # Import matplotlib for plotting

#################################
#################################
#  Here establish the Cut_off  ####
Freq_Cut=500 # @param {type:"slider", min:100, max:2000, step:50}
#################################
################################

#to get the butter signal paramters
b,a = signal.butter (4,Freq_Cut,btype='lowpass',fs=FreqMuestreo,output='ba')

# To filter using the filtfilt for zero-phase filtering to avoid time shifts
RegistroFiltrado = signal.filtfilt(b, a, registro.sweepY)

# Diplaying both graph separately
plt.figure(figsize=(12, 8))

# Original Signal
plt.subplot(2, 1, 1)
plt.plot(registro.sweepX, registro.sweepY)
plt.title("Original Signal")
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')

# Filtered Signal
plt.subplot(2, 1, 2)
plt.plot(registro.sweepX, RegistroFiltrado)
plt.title("Filtered Signal")
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')

plt.tight_layout() # Adjust layout to prevent overlapping titles/labels
plt.show()

SEGMENT TO ANALIZED

Select the interval in seconds to be analyzed




In [ ]:
import numpy as np
import plotly.graph_objects as go

####################################
####################################
#HERE I DEFINE THE SECTION OF THE SIGNAL I WILL USE FOR THE ANALYSIS#
Initial_Time = 7 # @param {type:"number", min:0, max:65}
Final_Time = 50 # @param {type:"number", min:0, max:65}
#####################################
#####################################

indice_inicial = (registro.sweepX >= Initial_Time).argmax()  #Finds the index for when the value in the array time(resgitro.sweepx) is the tiempoinicial
indice_final = (registro.sweepX <= Final_Time).sum() - 1  ##Finds the index for when the value in the array time(resgitro.sweepx) is the tiempofinal

#Cut both arrays
tiempo_cortado = registro.sweepX[indice_inicial:indice_final]
registro_filtrado_cortado = RegistroFiltrado[indice_inicial:indice_final]

## Here plotly is used instead of matplotlib to be able to interact with the plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=tiempo_cortado,y=registro_filtrado_cortado, mode='lines',name='Signal Cutted'))
fig.update_layout(title="Filtered and Cutted Signal",
                   xaxis_title='Time (s)',
                   yaxis_title='Amplitude (pA)')
fig.show()




# Stablishing Threshold
Determine the threshold for detecting EPSCs and IPSCs - like events.
The threhsold will be stablished in times the mean + and - the satndar devation (SD) of the signa, this depends on the noise of the signal but could be around 4X de SD.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

###############################################################
###############################################################
##### HERE DETERMINE HOW MANY TIMES THE SD I WILL USED TO STABLISH THRESHOLD
Times_SD = 3.8 # @param {type:"number"}

###############################################################
###############################################################

promediosenal = np.mean(registro_filtrado_cortado)
umbral = np.std(registro_filtrado_cortado)*Times_SD
umbralbajo = promediosenal - umbral
umbralalto = promediosenal + umbral

plt.figure(figsize=(12, 4))
plt.plot(tiempo_cortado, registro_filtrado_cortado, label='Filtered Signal', lw=0.8) # Plot the signal
plt.plot(tiempo_cortado, np.full_like(tiempo_cortado, promediosenal), 'r', label='Mean') # Plot the mean
plt.plot(tiempo_cortado, np.full_like(tiempo_cortado, umbralbajo), 'g', label='LowThreshold') # Plot mean + standard deviation
plt.plot(tiempo_cortado, np.full_like(tiempo_cortado, umbralalto), 'g', label='HighThreshold') # Plot mean - standard deviation
plt.title(f"Filtered Signal with Mean and Std (Cut from {Initial_Time}s to {Final_Time}s)")
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.show()



# Detection of EPSCs


In [ ]:
# @title Run to detect and display EPSCs detectd
import numpy as np
import matplotlib.pyplot as plt

indices_debajo_umbralbajo = np.where(registro_filtrado_cortado < umbralbajo)[0] ###This is to determine when the signals goes below threshold
#print(f"Number of points below umbralbajo: {len(indices_debajo_umbralbajo)}") #How many time this happens

##to be able to determine how many msecs we want to skip after the detection of the begining of an event we:
###############################################################
###############################################################
####here we determine how long in secods we will skip to count again an event###
Tiempo_de_espera_msec = 15
###############################################################
###############################################################


puntos_por_millisegundo = FreqMuestreo / 1000  # samples per millisecond
puntos_por_saltar = puntos_por_millisegundo * Tiempo_de_espera_msec  # samples to skip

indices_eventos_EPSC = []

if len(indices_debajo_umbralbajo) > 0:
    # Always keep the very first index as the start of an event
    indices_eventos_EPSC.append(indices_debajo_umbralbajo[0])

    # Iterate through the rest of the indices
    for i in range(1, len(indices_debajo_umbralbajo)):
        current_index = indices_debajo_umbralbajo[i]
        last_kept_index = indices_eventos_EPSC[-1]

        # If the current index is sufficiently far (at least min_separation_samples)
        # from the last kept index, consider it a new, distinct event onset.
        if (current_index - last_kept_index) >= puntos_por_saltar:
            indices_eventos_EPSC.append(current_index)

indices_eventos_EPSC = np.array(indices_eventos_EPSC)

#print(f"Original number of points below threshold: {len(indices_debajo_umbralbajo)}")
print(f"Number of EPSCs detected: {len(indices_eventos_EPSC)}")

plt.figure(figsize=(15, 6))
plt.plot(tiempo_cortado, registro_filtrado_cortado, label='Filtered Signal', lw=0.8)
plt.plot(tiempo_cortado, np.full_like(tiempo_cortado, umbralbajo), 'g', label='Low Threshold')

# Plot vertical lines for each detected event
for idx in indices_eventos_EPSC:
    event_time = tiempo_cortado[idx]
    plt.axvline(x=event_time, color='r', linewidth=0.2, label='_nolegend_' if idx != indices_eventos_EPSC[0] else 'Detected Event')

plt.title(f"Filtered Signal with Detected EPSPs (Cut from {Initial_Time}s to {Final_Time}s)")
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.show()

######################################################################
######################################################################
#HERE IS TO HAVE THE DURATION OF THE WINDOW TO SHOW EACH EPSCs in miliseconds###
ventana_ms = 60  # Duration of the window in milliseconds
######################################################################
######################################################################

ventana_s = ventana_ms / 1000  # Convert milliseconds to seconds
puntos_ventana = int(ventana_s * FreqMuestreo)  # Calculate number of data points

#To display all potential EPSCs on the same y-axis (amplitude) and be able to
# compare amplitudes I will get the minumum value of registro_filtrado_cortado
amplitudminima = np.min(registro_filtrado_cortado)
amplitudmaxima = np.max(registro_filtrado_cortado)

#Displaying each EPSCs to take the desicion
for i,eventos in enumerate(indices_eventos_EPSC):
    # Calculate start and end indices for the segments aound the EPSCs
    elprimero = max(0,eventos-puntos_ventana)
    elultimo = min(len(registro_filtrado_cortado),eventos+puntos_ventana)

    #extract the sgement around EPSCs
    segmento_tiempo = tiempo_cortado[elprimero:elultimo]
    segmento_senal = registro_filtrado_cortado[elprimero:elultimo]

    # Get the moment of the EPSCs
    momento_EPSC = tiempo_cortado[eventos]

    # Plot the individual EPSC
    plt.figure(figsize=(4, 2))
    plt.plot(segmento_tiempo, segmento_senal, color='blue')
    plt.axvline(x=momento_EPSC, color='red', linestyle='--')
    plt.title(f"EPSC {i+1} at {momento_EPSC:.3f} s")
    plt.xlabel('Time (s)')
    plt.ylim(amplitudminima, amplitudmaxima)
    plt.ylabel('Amplitude (pA)')
    plt.legend()
    plt.show()



# DISCARTING THE NO CONVINCING EPSCs
Looking at every single EPSCs displayed above, In the box below write the number of each descarted EPCSs separted by a , (example 2, 5, 9, 10).

This will discard the no convinving EPSCs.

If all are convincing leave the box in blanc and run the next section.

This section will display the new considered EPSCs and at the end will show the avarage trace, the average amplitude and the EPSCs frequency

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

######################################################################
######################################################################
# HERE YOU SHOULD LIST THE EPSCs THAT YOU WANT TO ELIMINATE
# For example, if you want to remove the 2nd and 5th detected EPSC, you would write:
# eventos_a_eliminar = [1, 4] (remember indices are 0-based)
######################################################################
######################################################################

Eliminated_EPSCs_events = "6,7,8,15,16,20" # @param {type:"string"}

# Convert the string input to a list of integers
# Handle empty string case to avoid errors if nothing is entered
if Eliminated_EPSCs_events.strip():
    eventos_a_eliminar = [int(x.strip()) for x in Eliminated_EPSCs_events.split(',')]
else:
    eventos_a_eliminar = []


#since the phyton array is 0-base I will create a new array with all minus 1
eventos_a_eliminar_zerobase = np.array(eventos_a_eliminar) - 1

#Creating a boolean mask to idenfied which elements will be elminated
mascara_EPSCs_validos = np.ones(len(indices_eventos_EPSC), dtype=bool) #Creates a mask with ones same size of indice_eventos_EPSC

for idx_a_eliminar in eventos_a_eliminar_zerobase:  #For loop that goes in all eventos_a_eliminar_zerobase
    if 0 <= idx_a_eliminar < len(indices_eventos_EPSC): #Asking if all elements to being eleminated actually exists in the EPSCs detected
        mascara_EPSCs_validos[idx_a_eliminar]= False

indices_eventos_EPSC_validos = indices_eventos_EPSC[mascara_EPSCs_validos]

#Displaying each EPSCs take as real
for i,eventos in enumerate(indices_eventos_EPSC_validos):
    # Calculate start and end indices for the segments aound the IPSCs
    elprimero = max(0,eventos-puntos_ventana)
    elultimo = min(len(registro_filtrado_cortado),eventos+puntos_ventana)

    #extract the sgement around EPSCs
    segmento_tiempo = tiempo_cortado[elprimero:elultimo]
    segmento_senal = registro_filtrado_cortado[elprimero:elultimo]

    # Get the moment of the EPSCs
    momento_EPSC = tiempo_cortado[eventos]

    # Plot the individual EPSC
    plt.figure(figsize=(4, 2))
    plt.plot(segmento_tiempo, segmento_senal, label='Segment', color='blue')
    plt.axvline(x=momento_EPSC, color='red', linestyle='--', label='Time of EPSC')
    plt.title(f"EPSC {i+1} at {momento_EPSC:.3f} s")
    plt.xlabel('Time (s)')
    plt.ylim(amplitudminima, amplitudmaxima)
    plt.ylabel('Amplitude (pA)')
    plt.legend()
    plt.show()


#To do the averages
#Calculating the duration of the segment
duracion_segmento=Final_Time-Initial_Time

#Calculating Frequency
frecuencia_EPSC = len(indices_eventos_EPSC_validos) / duracion_segmento

arreglo_EPSCs = []
#To do the average plot and all the pick amplitudes with sd
#We iterate on each EPSCs and take them
for k,eventos in enumerate(indices_eventos_EPSC_validos):
    # Calculate start and end indices for the segments aound the EPSCs
    elprimero = max(0,eventos-puntos_ventana)
    elultimo = min(len(registro_filtrado_cortado),eventos+puntos_ventana)

    #extract the sgement around EPSCs
    segmento_tiempo = tiempo_cortado[elprimero:elultimo]
    segmento_senal = registro_filtrado_cortado[elprimero:elultimo]

    arreglo_EPSCs.append(segmento_senal)

promedio_EPSC = np.mean(arreglo_EPSCs, axis=0)
desviacion_EPSC = np.std(arreglo_EPSCs, axis=0)

#Removing the offset of the promedio_EPSC
promedio_EPSC_sin_offset = promedio_EPSC - np.mean(promedio_EPSC[:int(puntos_por_millisegundo)*3]) #Considering only the fisrt 3 msec

# Plot
plt.figure(figsize=(8, 4))
plt.plot(segmento_tiempo, promedio_EPSC_sin_offset, label='mean EPSC', color='blue')
plt.plot(segmento_tiempo, promedio_EPSC_sin_offset-desviacion_EPSC, label='SD', color='red', lw=0.2)
plt.plot(segmento_tiempo, promedio_EPSC_sin_offset+desviacion_EPSC, color='red', lw=0.2)
plt.title(f"average EPSC")
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (pA)')
plt.legend()
#The last figure is dowload to be able to export the files on other software (Corel, PTT)
plt.savefig('EPSC_av.svg', format='svg') # High-quality vector format
files.download('EPSC_av.svg')
plt.show()

amplitud_promedio_EPSC = np.min(promedio_EPSC_sin_offset)

#Printing the results
print(f"EPSCs frequency:",frecuencia_EPSC)
print(f"Mean amplitude:",amplitud_promedio_EPSC)


# Detection of IPSCs

In [ ]:
# @title Run to detect and display IPSCs detectd
import numpy as np
import matplotlib.pyplot as plt

indices_arriba_umbralalto = np.where(registro_filtrado_cortado > umbralalto)[0] ###This is to determine when the signals goes above threshold

##to be able to determine how many msecs we want to skip after the detection of the begining of an event we:
###############################################################
###############################################################
####here we determine how long in secods we will skip to count again an event###
Tiempo_de_espera_msec = 15
###############################################################
###############################################################


puntos_por_millisegundo = FreqMuestreo / 1000  # samples per millisecond
puntos_por_saltar = puntos_por_millisegundo * Tiempo_de_espera_msec  # samples to skip

indices_eventos_IPSC = []

if len(indices_arriba_umbralalto) > 0:
    # Always keep the very first index as the start of an event
    indices_eventos_IPSC.append(indices_arriba_umbralalto[0])

    # Iterate through the rest of the indices
    for i in range(1, len(indices_arriba_umbralalto)):
        current_index = indices_arriba_umbralalto[i]
        last_kept_index = indices_eventos_IPSC[-1]

        # If the current index is sufficiently far (at least min_separation_samples)
        # from the last kept index, consider it a new, distinct event onset.
        if (current_index - last_kept_index) >= puntos_por_saltar:
            indices_eventos_IPSC.append(current_index)

indices_eventos_IPSC = np.array(indices_eventos_IPSC)

#print(f"Original number of points below threshold: {len(indices_debajo_umbralbajo)}")
print(f"Number of IPSCs detected: {len(indices_eventos_IPSC)}")

plt.figure(figsize=(15, 6))
plt.plot(tiempo_cortado, registro_filtrado_cortado, label='Filtered Signal', lw=0.8)
plt.plot(tiempo_cortado, np.full_like(tiempo_cortado, umbralalto), 'g', label='Low Threshold')

# Plot vertical lines for each detected event
for idx in indices_eventos_IPSC:
    event_time = tiempo_cortado[idx]
    plt.axvline(x=event_time, color='r', linewidth=0.2, label='_nolegend_' if idx != indices_eventos_IPSC[0] else 'Detected Event')

plt.title(f"Filtered Signal with Detected IPSPs (Cut from {Initial_Time}s to {Final_Time}s)")
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.show()

######################################################################
######################################################################
#HERE IS TO HAVE THE DURATION OF THE WINDOW TO SHOW EACH IPSCs in miliseconds###
ventana_ms = 60  # Duration of the window in milliseconds
######################################################################
######################################################################

ventana_s = ventana_ms / 1000  # Convert milliseconds to seconds
puntos_ventana = int(ventana_s * FreqMuestreo)  # Calculate number of data points

#To display all potential IPSCs on the same y-axis (amplitude) and be able to
# compare amplitudes I will get the minumum value of registro_filtrado_cortado
amplitudminima = np.min(registro_filtrado_cortado)
amplitudmaxima = np.max(registro_filtrado_cortado)

#Displaying each IPSCs to take the desicion
for i,eventos in enumerate(indices_eventos_IPSC):
    # Calculate start and end indices for the segments aound the IPSCs
    elprimero = max(0,eventos-puntos_ventana)
    elultimo = min(len(registro_filtrado_cortado),eventos+puntos_ventana)

    #extract the sgement around IPSCs
    segmento_tiempo = tiempo_cortado[elprimero:elultimo]
    segmento_senal = registro_filtrado_cortado[elprimero:elultimo]

    # Get the moment of the IPSCs
    momento_IPSC = tiempo_cortado[eventos]

    # Plot the individual IPSC
    plt.figure(figsize=(4, 2))
    plt.plot(segmento_tiempo, segmento_senal, color='blue')
    plt.axvline(x=momento_IPSC, color='red', linestyle='--')
    plt.title(f"IPSC {i+1} at {momento_IPSC:.3f} s")
    plt.xlabel('Time (s)')
    plt.ylim(amplitudminima, amplitudmaxima)
    plt.ylabel('Amplitude (pA)')
    plt.legend()
    plt.show()

# DISCARTING THE NO CONVINCING IPSCs
Looking at every single IPSCs displayed above, In the box below write the number of each descarted IPCSs separted by a , (example 2, 5, 9, 10).

This will discard the no convinving IPSCs.

If all are convincing leave the box in blanc and run the next section.

This section will display the new considered IPSCs and at the end will show the avarage trace, the averae amplitude and the IPSCs frequency

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

######################################################################
######################################################################
# HERE YOU SHOULD LIST THE IPSCs THAT YOU WANT TO ELIMINATE
# For example, if you want to remove the 2nd and 5th detected IPSC, you would write:
# eventos_a_eliminar = [1, 4] (remember indices are 0-based)
######################################################################
######################################################################

Eliminated_IPSCs_events = "1,7" # @param {type:"string"}

# Convert the string input to a list of integers
# Handle empty string case to avoid errors if nothing is entered
if Eliminated_IPSCs_events.strip():
    eventos_a_eliminar = [int(x.strip()) for x in Eliminated_IPSCs_events.split(',')]
else:
    eventos_a_eliminar = []


#since the phyton array is 0-base I will create a new array with all minus 1
eventos_a_eliminar_zerobase = np.array(eventos_a_eliminar) - 1

#Creating a boolean mask to idenfied which elements will be elminated
mascara_IPSPs_validos = np.ones(len(indices_eventos_IPSC), dtype=bool) #Creates a mask with ones same size of indice_eventos_IPSC

for idx_a_eliminar in eventos_a_eliminar_zerobase:  #For loop that goes in all eventos_a_eliminar_zerobase
    if 0 <= idx_a_eliminar < len(indices_eventos_IPSC): #Asking if all elements to being eleminated actually exists in the IPSCs detected
        mascara_IPSPs_validos[idx_a_eliminar]= False

indices_eventos_IPSC_validos = indices_eventos_IPSC[mascara_IPSPs_validos]

#Displaying each IPSCs take as real
for i,eventos in enumerate(indices_eventos_IPSC_validos):
    # Calculate start and end indices for the segments aound the IPSCs
    elprimero = max(0,eventos-puntos_ventana)
    elultimo = min(len(registro_filtrado_cortado),eventos+puntos_ventana)

    #extract the sgement around IPSCs
    segmento_tiempo = tiempo_cortado[elprimero:elultimo]
    segmento_senal = registro_filtrado_cortado[elprimero:elultimo]

    # Get the moment of the IPSCs
    momento_IPSC = tiempo_cortado[eventos]

    # Plot the individual IPSC
    plt.figure(figsize=(4, 2))
    plt.plot(segmento_tiempo, segmento_senal, label='Segment', color='blue')
    plt.axvline(x=momento_IPSC, color='red', linestyle='--', label='Time of IPSC')
    plt.title(f"IPSC {i+1} at {momento_IPSC:.3f} s")
    plt.xlabel('Time (s)')
    plt.ylim(amplitudminima, amplitudmaxima)
    plt.ylabel('Amplitude (pA)')
    plt.legend()
    plt.show()



#To do the averages
#Calculating the duration of the segment
duracion_segmento=Final_Time-Initial_Time

#Calculating Frequency
frecuencia_IPSC = len(indices_eventos_IPSC_validos) / duracion_segmento

arreglo_IPSCs = []
#To do the average plot and all the pick amplitudes with sd
#We iterate on each IPSCs and take them
for k,eventos in enumerate(indices_eventos_IPSC_validos):
    # Calculate start and end indices for the segments aound the IPSCs
    elprimero = max(0,eventos-puntos_ventana)
    elultimo = min(len(registro_filtrado_cortado),eventos+puntos_ventana)

    #extract the sgement around IPSCs
    segmento_tiempo = tiempo_cortado[elprimero:elultimo]
    segmento_senal = registro_filtrado_cortado[elprimero:elultimo]

    arreglo_IPSCs.append(segmento_senal)

promedio_IPSC = np.mean(arreglo_IPSCs, axis=0)
desviacion_IPSC = np.std(arreglo_IPSCs, axis=0)

#Removing the offset of the promedio_IPSC
promedio_IPSC_sin_offset = promedio_IPSC - np.mean(promedio_IPSC[:int(puntos_por_millisegundo)*3]) #Considering only the fisrt 3 msec

# Plot
plt.figure(figsize=(8, 4))
plt.plot(segmento_tiempo, promedio_IPSC_sin_offset, label='mean IPSC', color='blue')
plt.plot(segmento_tiempo, promedio_IPSC_sin_offset-desviacion_IPSC, label='SD', color='red', lw=0.2)
plt.plot(segmento_tiempo, promedio_IPSC_sin_offset+desviacion_IPSC, color='red', lw=0.2)
plt.title(f"average IPSC")
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (pA)')
plt.legend()
#The last figure is dowload to be able to export the files on other software (Corel, PTT)
plt.savefig('IPSC_av.svg', format='svg') # High-quality vector format
files.download('IPSC_av.svg')
plt.show()

amplitud_promedio_IPSC = np.max(promedio_IPSC_sin_offset)

#Printing the results
print(f"IPSCs frequency:",frecuencia_IPSC)
print(f"Mean amplitude:",amplitud_promedio_IPSC)
